# Qdrant: индексация документов и поиск

Здесь используем все Markdown-документы из `data/knowledge_base` и модель эмбеддингов `BAAI/bge-m3`, как в основном ноутбуке проекта. Документы разбиваются на фрагменты; каждый фрагмент записывается в Qdrant вместе с вектором и именем файла.

Из корня проекта: `uv sync` и `docker compose up -d qdrant`. При первом запуске модель может скачиваться. Ноутбук использует отдельную коллекцию `robotex_vector_search_demo`.

In [1]:
import os
from pathlib import Path
from uuid import NAMESPACE_URL, uuid5

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

DATA_DIR = Path("../data/knowledge_base")
COLLECTION = "robotex_vector_search_demo"
MODEL = "BAAI/bge-m3"
DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Модель: {MODEL}, устройство: {DEVICE}")

Модель: BAAI/bge-m3, устройство: mps


## Загрузка и разбиение документов

In [2]:
files = sorted(DATA_DIR.rglob("*.md"))
if not files:
    raise FileNotFoundError(f"Документы не найдены: {DATA_DIR.resolve()}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200,
    separators=["\n## ", "\n\n", "\n", " ", ""],
)
chunks = []
for path in files:
    for i, text in enumerate(splitter.split_text(path.read_text(encoding="utf-8"))):
        chunks.append({"text": text, "source": str(path.relative_to(DATA_DIR)), "chunk": i})

print(f"Документов: {len(files)}, фрагментов: {len(chunks)}")

Документов: 36, фрагментов: 79


## Векторизация и запись в Qdrant

In [3]:
embeddings = HuggingFaceEmbeddings(
    model_name=MODEL,
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)
client = QdrantClient(url="http://localhost:6333", timeout=120)

# Отдельная коллекция с размерностью BAAI/bge-m3.
if not client.collection_exists(COLLECTION):
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=models.VectorParams(size=1024, distance=models.Distance.COSINE),
    )

# Стабильные ID позволяют обновлять фрагменты при повторном запуске.
for start in range(0, len(chunks), 16):
    batch = chunks[start:start + 16]
    vectors = embeddings.embed_documents([chunk["text"] for chunk in batch])
    points = [
        models.PointStruct(
            id=str(uuid5(NAMESPACE_URL, f"{chunk['source']}:{chunk['chunk']}")),
            vector=vector,
            payload=chunk,
        )
        for chunk, vector in zip(batch, vectors)
    ]
    client.upsert(collection_name=COLLECTION, points=points, wait=True)
    print(f"Записано: {start + len(batch)}/{len(chunks)}")

print("Всего в Qdrant:", client.count(collection_name=COLLECTION, exact=True).count)

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

... (2 outputs left out)

Записано: 32/79


Записано: 48/79


Записано: 64/79


Записано: 79/79
Всего в Qdrant: 79


## Поиск

Запрос векторизуется той же моделью. Score — косинусное сходство: чем выше, тем ближе векторы.

In [4]:
def search(query, limit=3):
    results = client.query_points(
        collection_name=COLLECTION,
        query=embeddings.embed_query(query),
        limit=limit,
        with_payload=True,
    ).points
    for point in results:
        print(f"Score: {point.score:.4f} | {point.payload['source']} | фрагмент {point.payload['chunk']}")
        print(point.payload["text"][:500], "\n")

In [5]:
search("Какие правила удалённой работы?")

Score: 0.6784 | HR_policy_2024.md | фрагмент 1
## III. Основные положения политики

1. **Квалификация и соответствие:** Для обеспечения эффективности работы, всем сотрудникам, желающим работать удаленно, требуется иметь необходимый опыт и квалификацию для выполнения обязанностей.

2. **Технические требования:** Для обеспечения качества работы и связи с коллегами, всем сотрудникам, работающим удаленно, предоставляется набор необходимого оборудования и программного обеспечения.

3. **Профессиональная ответственность:** Всем сотрудникам, работа 

Score: 0.6657 | HR_policy_2024.md | фрагмент 0
# Политика Удаленной Работы (Архив 2024)

**ВНИМАНИЕ:** Этот документ относится к Архиву 2024 и предназначен для ссылки на историческую информацию о политике удаленной работы в корпорации РобоТех.

## I. Введение

В соответствии с измененными требованиями к работе и жизни в обществе, в нашей корпорации РобоТех был разработан и принят документ о политике удаленной работы. Целью данного документа являе

In [6]:
search("Кто отвечает за проект Atlas и какие системы он использует?")

Score: 0.6807 | Project_Atlas_Graph_2026.md | фрагмент 0
---
year: 2026
category: Projects
department: AI_Platform
doc_type: project_brief
access_group: engineering
graph_ready: true
---
# Проект Atlas: графовая навигация по корпоративным знаниям

Проект Atlas строит Graph RAG для корпоративных документов РобоТех. Цель проекта: связать документы, сотрудников, команды, политики и технические системы в одном property graph.

Мария Соколова является владельцем бизнес-требований проекта Atlas. Иван Петров является техническим лидом проекта Atlas. Анна Ким 

Score: 0.6243 | Project_Atlas_Graph_2026.md | фрагмент 1
Команда AI Platform отвечает за RAG pipeline. Команда Security отвечает за требования ZTA-17. Команда PeopleOps отвечает за политику HR-REMOTE-2026.

Ключевые связи:
- Мария Соколова owns HR-REMOTE-2026.
- Иван Петров leads Project Atlas.
- Project Atlas uses Neo4j.
- Project Atlas uses Qdrant.
- Project Atlas follows ZTA-17. 

Score: 0.5600 | Meeting_Atlas_Security_2026.md | фраг